# EDA — Part A: Basic Inspection

**Goal:** Know the data before modelling it.  
This notebook characterises every FER-2013 CSV: schema differences between files,
shape, dtypes, missing values, pixel value ranges, label distribution, and Usage split counts.

Every anomaly found here feeds directly into the cleaning and preprocessing stages
(`config.yaml → cleaning.*` and `preprocessing.*`).

**File inventory** (all live in `data/` after the download stage):

| File | Columns | Role in pipeline |
|---|---|---|
| `train.csv` | `emotion`, `pixels` | Flat training rows — no split column |
| `test.csv` | `pixels` | Unlabelled; not used by this pipeline |
| `test_with_emotions.csv` | `emotion`, `pixels` | Held-out evaluation set |
| `icml_face_data.csv` | `emotion`, `usage`, `pixels` | **Primary CSV** — has the split column |
| `fer2013/fer2013.csv` | `emotion`, `pixels`, `usage` | Same schema as `icml_face_data.csv` |

`Fer2013Fetcher` targets `icml_face_data.csv` (`config.yaml → data.primary_csv`)  
because it is the only top-level file with a `usage` column to separate
Training / PublicTest / PrivateTest.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path().resolve().parent  # notebooks/ → project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config
from src.emotion_detector.utils.logging import setup_logging
from src.emotion_detector.data.fer2013 import Fer2013Fetcher

cfg = load_config(ROOT / "config.yaml")

# config.yaml stores relative paths (e.g. "data/") intended to be resolved
# from the project root. When running inside notebooks/, the CWD is wrong,
# so we make every path absolute here before any module reads cfg["paths"].
for key in cfg["paths"]:
    cfg["paths"][key] = str(ROOT / cfg["paths"][key])

setup_logging(cfg)

DATA_DIR = Path(cfg["paths"]["data_dir"])

FILES = {
    "primary"  : DATA_DIR / cfg["data"]["primary_csv"],   # icml_face_data.csv
    "train"    : DATA_DIR / cfg["data"]["train_csv"],      # train.csv
    "test_eval": DATA_DIR / cfg["data"]["test_csv"],       # test_with_emotions.csv
}

for label, path in FILES.items():
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"{status}  {label:12} → {path}")

2026-07-04 12:55:13 | INFO     | src.emotion_detector.utils.logging:61 — Logging initialised — level=INFO, log_dir=/Users/mohammad.kheirkhah/Desktop/emotions-detecto/logs


✓  primary      → /Users/mohammad.kheirkhah/Desktop/emotions-detecto/data/icml_face_data.csv
✓  train        → /Users/mohammad.kheirkhah/Desktop/emotions-detecto/data/train.csv
✓  test_eval    → /Users/mohammad.kheirkhah/Desktop/emotions-detecto/data/test_with_emotions.csv


## 1. Schema survey — what columns does each file actually have?

Each file has a different schema. This cell loads just the header row
(0 data rows) so we can compare without reading 35 k rows.

In [2]:
for label, path in FILES.items():
    header = pd.read_csv(path, nrows=0)
    print(f"{path.name}")
    print(f"  columns : {list(header.columns)}")
    print()

icml_face_data.csv
  columns : ['emotion', ' Usage', ' pixels']

train.csv
  columns : ['emotion', 'pixels']

test_with_emotions.csv
  columns : ['Unnamed: 0', 'emotion', 'pixels']



## 2. Load the primary CSV (`icml_face_data.csv`)

This is the file that drives `Fer2013Fetcher` — it has all rows plus the `usage`
column that separates Training / PublicTest / PrivateTest.

In [3]:
df = pd.read_csv(FILES["primary"])

# Strip leading/trailing whitespace from column names — the real CSV
# ships headers like ' Usage' and ' pixels' with a leading space.
df.columns = df.columns.str.strip()

# Normalise the split column: real data ships as lowercase 'usage'
usage_col = next((c for c in df.columns if c.lower() == "usage"), None)
if usage_col and usage_col != "Usage":
    df = df.rename(columns={usage_col: "Usage"})

print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
df.head()

Shape   : (35887, 3)
Columns : ['emotion', 'Usage', 'pixels']


,emotion,Usage,pixels
0,0,Training,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,Training,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,Training,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,Training,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,Training,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


## 3. Schema — dtypes

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35887 entries, 0 to 35886
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   emotion  35887 non-null  int64
 1   Usage    35887 non-null  str  
 2   pixels   35887 non-null  str  
dtypes: int64(1), str(2)
memory usage: 287.8 MB


In [5]:
print("Dtypes:")
print(df.dtypes)
print()
print("Expected:")
print("  emotion  → int64             (label 0–6)")
print("  pixels   → object or str     (space-separated pixel string; str = StringDtype in pandas 3.x)")
print("  Usage    → object or str     (Training / PublicTest / PrivateTest)")
print()
print("Note: 'object' and 'str' both mean string data.")
print("      pandas 3.x uses StringDtype (displayed as 'str') instead of the")
print("      legacy object dtype. Both are handled identically by str methods and np.fromstring.")

Dtypes:
emotion    int64
Usage        str
pixels       str
dtype: object

Expected:
  emotion  → int64             (label 0–6)
  pixels   → object or str     (space-separated pixel string; str = StringDtype in pandas 3.x)
  Usage    → object or str     (Training / PublicTest / PrivateTest)

Note: 'object' and 'str' both mean string data.
      pandas 3.x uses StringDtype (displayed as 'str') instead of the
      legacy object dtype. Both are handled identically by str methods and np.fromstring.


## 4. Summary statistics

In [6]:
df.describe(include="all")

,emotion,Usage,pixels
count,35887.000000,35887,35887
unique,NaN,3,34034
top,NaN,Training,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ...
freq,NaN,28709,12
mean,3.323265,NaN,NaN
std,1.873819,NaN,NaN
min,0.000000,NaN,NaN
25%,2.000000,NaN,NaN
50%,3.000000,NaN,NaN
75%,5.000000,NaN,NaN


## 5. Missing value audit

In [7]:
nan_counts = df.isna().sum()
print("NaN counts per column:")
print(nan_counts)
print()
if nan_counts.sum() == 0:
    print("✓ No NaN values found.")
else:
    print(f"⚠ {nan_counts.sum()} NaN values — see cleaning stage.")

NaN counts per column:
emotion    0
Usage      0
pixels     0
dtype: int64

✓ No NaN values found.


In [8]:
empty_pixels = df["pixels"].str.strip().eq("").sum()
print(f"Empty / whitespace-only pixels rows: {empty_pixels}")
if empty_pixels == 0:
    print("✓ No empty pixel strings.")

Empty / whitespace-only pixels rows: 0
✓ No empty pixel strings.


## 6. Pixel-count sanity check

Every row must encode exactly **2 304 values** (48 × 48).

In [9]:
pixel_counts = df["pixels"].str.split().str.len()

print("Pixel-count distribution:")
print(pixel_counts.value_counts().sort_index())
print()

bad_rows = df[pixel_counts != 2304]
print(f"Rows with pixel count ≠ 2304: {len(bad_rows)}")
if len(bad_rows) == 0:
    print("✓ All rows have exactly 2304 pixel values.")
else:
    print("⚠ Malformed rows:")
    print(bad_rows[["emotion", "Usage"]].assign(pixel_count=pixel_counts[bad_rows.index]))

Pixel-count distribution:
pixels
2304    35887
Name: count, dtype: int64

Rows with pixel count ≠ 2304: 0
✓ All rows have exactly 2304 pixel values.


## 7. Pixel intensity statistics

`Fer2013Fetcher` parses the pixel strings into `(N, 48, 48)` uint8 arrays.
We compute intensity stats on the Training split.

In [10]:
fetcher = Fer2013Fetcher(cfg)

images_train, labels_train = fetcher.fetch("Training")
images_val,   labels_val   = fetcher.fetch("PublicTest")
images_test,  labels_test  = fetcher.fetch("PrivateTest")

print(f"Training   — images: {images_train.shape}, labels: {labels_train.shape}")
print(f"Validation — images: {images_val.shape},   labels: {labels_val.shape}")
print(f"Test       — images: {images_test.shape},  labels: {labels_test.shape}")

2026-07-04 12:55:27 | INFO     | src.emotion_detector.data.fer2013:89 — Loading split 'Training': 28,709 rows from icml_face_data.csv
2026-07-04 12:55:28 | INFO     | src.emotion_detector.data.fer2013:104 — Split 'Training' ready — images (28709, 48, 48) uint8, labels (28709,)
2026-07-04 12:55:31 | INFO     | src.emotion_detector.data.fer2013:89 — Loading split 'PublicTest': 3,589 rows from icml_face_data.csv
2026-07-04 12:55:31 | INFO     | src.emotion_detector.data.fer2013:104 — Split 'PublicTest' ready — images (3589, 48, 48) uint8, labels (3589,)
2026-07-04 12:55:33 | INFO     | src.emotion_detector.data.fer2013:89 — Loading split 'PrivateTest': 3,589 rows from icml_face_data.csv
2026-07-04 12:55:33 | INFO     | src.emotion_detector.data.fer2013:104 — Split 'PrivateTest' ready — images (3589, 48, 48) uint8, labels (3589,)


Training   — images: (28709, 48, 48), labels: (28709,)
Validation — images: (3589, 48, 48),   labels: (3589,)
Test       — images: (3589, 48, 48),  labels: (3589,)


In [11]:
px = images_train.astype(np.float32)
print("Training pixel intensity statistics:")
print(f"  dtype   : {images_train.dtype}")
print(f"  min     : {px.min():.1f}")
print(f"  max     : {px.max():.1f}")
print(f"  mean    : {px.mean():.2f}")
print(f"  std     : {px.std():.2f}")
print(f"  median  : {np.median(px):.1f}")
print()
if px.min() >= 0 and px.max() <= 255:
    print("✓ All pixel values in valid range [0, 255].")
else:
    print("⚠ Out-of-range pixel values detected.")

Training pixel intensity statistics:
  dtype   : uint8
  min     : 0.0
  max     : 255.0
  mean    : 129.47
  std     : 65.03
  median  : 134.0

✓ All pixel values in valid range [0, 255].


## 8. Label distribution

In [12]:
EMOTION_LABELS = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad",     5: "Surprise", 6: "Neutral",
}

label_counts = df["emotion"].value_counts().sort_index()
label_pct    = label_counts / len(df) * 100

print("Label distribution (full icml_face_data.csv):")
print(f"{'Code':<6} {'Label':<12} {'Count':>7} {'%':>7}")
print("-" * 36)
for code, count in label_counts.items():
    print(f"{code:<6} {EMOTION_LABELS[code]:<12} {count:>7,} {label_pct[code]:>6.1f}%")
print("-" * 36)
print(f"{'Total':<18} {len(df):>7,} {'100.0%':>7}")
print()

out_of_range = df[~df["emotion"].between(0, 6)]
print(f"Out-of-range labels : {len(out_of_range)}")
if len(out_of_range) == 0:
    print("✓ All labels in [0, 6].")

Label distribution (full icml_face_data.csv):
Code   Label          Count       %
------------------------------------
0      Angry          4,953   13.8%
1      Disgust          547    1.5%
2      Fear           5,121   14.3%
3      Happy          8,989   25.0%
4      Sad            6,077   16.9%
5      Surprise       4,002   11.2%
6      Neutral        6,198   17.3%
------------------------------------
Total               35,887  100.0%

Out-of-range labels : 0
✓ All labels in [0, 6].


## 9. Usage split counts

In [13]:
usage_counts = df["Usage"].value_counts()
usage_pct    = usage_counts / len(df) * 100

print("Usage split distribution:")
print(f"{'Split':<16} {'Count':>7} {'%':>7}")
print("-" * 33)
for split, count in usage_counts.items():
    print(f"{split:<16} {count:>7,} {usage_pct[split]:>6.1f}%")
print("-" * 33)
print(f"{'Total':<16} {len(df):>7,} {'100.0%':>7}")
print()

unexpected = set(df["Usage"].unique()) - {"Training", "PublicTest", "PrivateTest"}
if not unexpected:
    print("✓ Only expected Usage values present.")
else:
    print(f"⚠ Unexpected Usage values: {unexpected}")

Usage split distribution:
Split              Count       %
---------------------------------
Training          28,709   80.0%
PublicTest         3,589   10.0%
PrivateTest        3,589   10.0%
---------------------------------
Total             35,887  100.0%

✓ Only expected Usage values present.


## 10. Per-split label distribution

Checks whether class imbalance is consistent across splits.

In [14]:
split_label = (
    df.groupby(["Usage", "emotion"])
    .size()
    .unstack(fill_value=0)
    .rename(columns=EMOTION_LABELS)
)
split_label

emotion,Angry,Disgust,Fear,Happy,Sad,Surprise,Neutral
Usage,,,,,,,
PrivateTest,491,55,528,879,594,416,626
PublicTest,467,56,496,895,653,415,607
Training,3995,436,4097,7215,4830,3171,4965


## 11. train.csv schema check

`train.csv` has only `emotion` and `pixels` — no `Usage` column.
It contains all training rows as a flat file.  
We confirm its shape and verify it has no NaN.

In [15]:
df_train = pd.read_csv(FILES["train"])
df_train.columns = df_train.columns.str.strip()
print(f"train.csv — shape: {df_train.shape}")
print(f"Columns  : {list(df_train.columns)}")
print(f"NaN      : {df_train.isna().sum().to_dict()}")
df_train.head(3)

train.csv — shape: (28709, 2)
Columns  : ['emotion', 'pixels']
NaN      : {'emotion': 0, 'pixels': 0}


,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...


## 12. test_with_emotions.csv schema check

This is the final held-out evaluation set — touched only once, after training.

In [16]:
df_eval = pd.read_csv(FILES["test_eval"])
df_eval.columns = df_eval.columns.str.strip()
print(f"test_with_emotions.csv — shape: {df_eval.shape}")
print(f"Columns  : {list(df_eval.columns)}")
print(f"NaN      : {df_eval.isna().sum().to_dict()}")
df_eval.head(3)

test_with_emotions.csv — shape: (7178, 3)
Columns  : ['Unnamed: 0', 'emotion', 'pixels']
NaN      : {'Unnamed: 0': 0, 'emotion': 0, 'pixels': 0}


,Unnamed: 0,emotion,pixels
0,0,0,254 254 254 254 254 249 255 160 2 58 53 70 77 ...
1,1,1,156 184 198 202 204 207 210 212 213 214 215 21...
2,2,4,69 118 61 60 96 121 103 87 103 88 70 90 115 12...


## 13. Findings summary

Results recorded from a full run on the actual dataset (35,887 rows in
`icml_face_data.csv`). Each anomaly that requires a fix becomes a named,
switchable option in `config.yaml` (CONTRIBUTING.md §3 — Ablation-Driven
Architecture); the analysis cells live in the cleaning issues and EDA-c (#13).

### Health checks (passed)

| Check | Result | Action |
|---|---|---|
| NaN values in `icml_face_data.csv` | 0 | none |
| Empty / whitespace pixel strings | 0 | none |
| Pixel count ≠ 2304 | 0 — all 35,887 rows have exactly 2304 | none |
| Pixel range [0, 255] | min 0, max 255 (uint8) ✓ | see caveat below |
| Label range [0, 6] | all in range, 0 out-of-range | none |
| Unexpected `Usage` values | none — only Training/PublicTest/PrivateTest | none |
| Split proportions | 80 % / 10 % / 10 % (28,709 / 3,589 / 3,589) | none |

### Anomalies found (feed later issues)

| Finding | Evidence | Where it's handled |
|---|---|---|
| **Class imbalance** | Disgust 1.5 % (547) vs Happy 25 % (8,989) — 16× gap; consistent across all three splits | `cleaning.imbalance_strategy: class_weight`; evaluate with macro-F1 + per-class recall, not accuracy (`evaluation.metrics`) |
| **Constant (all-zero) images** | `describe()` top pixel value is an all-zero string with `freq = 12` | cleaning filter: drop images with `std == 0` (generalises all-black / all-white / all-constant) → #13 + cleaning issue |
| **Duplicate images** | 34,034 unique pixel strings out of 35,887 → ~1,853 duplicates | `cleaning.remove_duplicates: true`; **must also check train↔test leakage** (same image in two splits inflates test accuracy — CONTRIBUTING §8) |
| **Slight left-skew of intensities** | mean 129.47 < median 134.0 → tail toward dark; healthy for face crops | quantify with boxplot + `scipy.stats.skew`/`kurtosis` in #13 (EDA-c) |
| **`test_with_emotions.csv` phantom column** | has `Unnamed: 0` (a saved pandas index) | load with `pd.read_csv(..., index_col=0)` |
| **`test_with_emotions.csv` size** | 7,178 rows = 3,589 + 3,589 = PublicTest + PrivateTest combined | it is the *full* held-out test set, not a separate one — document in `data.md` to avoid double-counting |

### Caveat — pixel-range validation must precede the uint8 cast

`Fer2013Fetcher` parses pixels with `np.fromstring(..., dtype=np.uint8)`. Any
out-of-range value is **silently wrapped** before we can see it (`300 → 44`,
since 300 mod 256 = 44). For FER this is theoretical (source was genuine 8-bit
images), but a defensive range filter in the cleaning stage must parse as
`int16` first, validate `0 ≤ v ≤ 255`, **then** cast to `uint8`.

### Candidate "bad row" filters (future `cleaning.*` ablation options)

| Filter | Rule | Catches |
|---|---|---|
| Constant | `std == 0` | all-black / all-white / all-gray |
| Low variance | `std < τ` | near-blank / fogged frames |
| Low dynamic range | `max − min < τ` | flat, washed-out images |
| Over/under-exposed | `>90 %` pixels at exactly 0 or 255 | blown-out / crushed images |
| Low entropy | histogram Shannon entropy `< τ` | information-poor images |